# Investigação de APIs para Sistema de Análise de Fundos

**Objetivo**: Investigar APIs oficiais disponíveis para coleta de dados do sistema de análise de fundos de investimento.

**Escopo de Investigação:**
1. APIs BCB/Tesouro para benchmarks (SELIC, CDI, Tesouro Direto)
2. APIs CVM para dados oficiais de fundos
3. APIs/RSS para notícias Nubank
4. Validação de estruturas de dados

**Data**: 2026-03-27  
**Responsável**: Mr. Robot (Arquiteto)

## 1. Setup e Dependências

In [ ]:
import requests
import pandas as pd
import json
from datetime import datetime, timedelta
import feedparser
from pprint import pprint

## 2. API BCB - Banco Central do Brasil

Investigando APIs para SELIC, CDI e outros benchmarks.

In [ ]:
# BCB API Base URL
BCB_BASE_URL = "https://api.bcb.gov.br/dados/serie/bcdata.sgs"

# Códigos das séries do BCB
SERIES_BCB = {
    'SELIC': 432,  # Taxa SELIC
    'CDI': 12,     # Taxa CDI
    'IPCA': 433    # IPCA (para referência)
}

def test_bcb_api(serie_code, serie_name):
    """Testa API do BCB para uma série específica"""
    print(f"\n=== Testando {serie_name} (Código: {serie_code}) ===")
    
    # Últimos 30 dias
    end_date = datetime.now().strftime('%d/%m/%Y')
    start_date = (datetime.now() - timedelta(days=30)).strftime('%d/%m/%Y')
    
    url = f"{BCB_BASE_URL}/{serie_code}/dados?formato=json&dataInicial={start_date}&dataFinal={end_date}"
    
    try:
        response = requests.get(url, timeout=10)
        print(f"Status: {response.status_code}")
        
        if response.status_code == 200:
            data = response.json()
            print(f"Registros encontrados: {len(data)}")
            
            if data:
                print("Estrutura dos dados:")
                pprint(data[0])
                
                if len(data) > 1:
                    print(f"Último registro:")
                    pprint(data[-1])
                    
            return data
        else:
            print(f"Erro: {response.text}")
            return None
            
    except Exception as e:
        print(f"Erro na requisição: {str(e)}")
        return None

In [ ]:
# Testando todas as séries do BCB
bcb_results = {}

for serie_name, serie_code in SERIES_BCB.items():
    result = test_bcb_api(serie_code, serie_name)
    bcb_results[serie_name] = result

## 3. API Tesouro Direto

Investigando APIs para dados do Tesouro Direto.

In [ ]:
# Tesouro Direto API
TESOURO_BASE_URL = "https://www.tesourotransparente.gov.br/ckan/api/3/action"

def test_tesouro_api():
    """Testa API do Tesouro Direto"""
    print("\n=== Testando API Tesouro Direto ===")
    
    # Listar datasets disponíveis
    url = f"{TESOURO_BASE_URL}/package_list"
    
    try:
        response = requests.get(url, timeout=10)
        print(f"Status: {response.status_code}")
        
        if response.status_code == 200:
            data = response.json()
            datasets = data.get('result', [])
            
            print(f"Datasets disponíveis: {len(datasets)}")
            
            # Procurar por datasets relacionados ao Tesouro Direto
            tesouro_datasets = [d for d in datasets if 'tesouro' in d.lower() or 'direto' in d.lower()]
            
            print(f"Datasets relacionados ao Tesouro Direto:")
            for dataset in tesouro_datasets[:10]:  # Primeiros 10
                print(f"- {dataset}")
                
            return datasets
        else:
            print(f"Erro: {response.text}")
            return None
            
    except Exception as e:
        print(f"Erro na requisição: {str(e)}")
        return None

In [ ]:
tesouro_datasets = test_tesouro_api()

## 4. APIs CVM - Comissão de Valores Mobiliários

Investigando APIs para dados oficiais de fundos.

In [ ]:
# CVM API para dados de fundos
CVM_BASE_URL = "http://dados.cvm.gov.br/dados"

def test_cvm_api():
    """Testa API da CVM para dados de fundos"""
    print("\n=== Testando API CVM ===")
    
    # Tentar acessar dados de fundos
    endpoints_to_test = [
        "/FI/CAD/DADOS/cad_fi.csv",
        "/FI/DOC/INF_DIARIO/DADOS",
        "/FI/DOC/COMPL/DADOS"
    ]
    
    results = {}
    
    for endpoint in endpoints_to_test:
        print(f"\nTestando endpoint: {endpoint}")
        url = f"{CVM_BASE_URL}{endpoint}"
        
        try:
            # Para endpoints de dados, fazer HEAD request primeiro
            response = requests.head(url, timeout=10)
            print(f"Status HEAD: {response.status_code}")
            print(f"Headers: {dict(response.headers)}")
            
            results[endpoint] = {
                'status': response.status_code,
                'headers': dict(response.headers)
            }
            
        except Exception as e:
            print(f"Erro na requisição: {str(e)}")
            results[endpoint] = {'error': str(e)}
            
    return results

In [ ]:
cvm_results = test_cvm_api()

## 5. RSS Feeds e APIs de Notícias Nubank

Investigando fontes de notícias sobre Nubank.

In [ ]:
def test_news_sources():
    """Testa fontes de notícias sobre Nubank"""
    print("\n=== Testando Fontes de Notícias ===")
    
    news_sources = {
        'Google News RSS': 'https://news.google.com/rss/search?q=nubank&hl=pt-BR&gl=BR&ceid=BR:pt-419',
        'InfoMoney RSS': 'https://www.infomoney.com.br/rss/',
        'Valor Econômico RSS': 'https://valor.globo.com/rss'
    }
    
    results = {}
    
    for source_name, url in news_sources.items():
        print(f"\n--- Testando {source_name} ---")
        
        try:
            if 'rss' in url.lower() or 'feed' in url.lower():
                # Testar RSS feed
                feed = feedparser.parse(url)
                
                print(f"Status: {feed.status if hasattr(feed, 'status') else 'N/A'}")
                print(f"Título do feed: {feed.feed.get('title', 'N/A')}")
                print(f"Entradas encontradas: {len(feed.entries)}")
                
                if feed.entries:
                    print(f"Primeira entrada:")
                    entry = feed.entries[0]
                    print(f"  Título: {entry.get('title', 'N/A')}")
                    print(f"  Link: {entry.get('link', 'N/A')}")
                    print(f"  Data: {entry.get('published', 'N/A')}")
                    
                results[source_name] = {
                    'type': 'rss',
                    'status': feed.status if hasattr(feed, 'status') else 'success',
                    'entries_count': len(feed.entries),
                    'feed_title': feed.feed.get('title', 'N/A')
                }
            else:
                # Testar endpoint HTTP simples
                response = requests.head(url, timeout=10)
                print(f"Status: {response.status_code}")
                
                results[source_name] = {
                    'type': 'http',
                    'status': response.status_code
                }
                
        except Exception as e:
            print(f"Erro: {str(e)}")
            results[source_name] = {'error': str(e)}
            
    return results

In [ ]:
news_results = test_news_sources()

## 6. Análise da API MaisRetorno (Investigação Adicional)

Verificando se existe API pública do MaisRetorno.

In [ ]:
def investigate_maisretorno():
    """Investiga possíveis APIs do MaisRetorno"""
    print("\n=== Investigando MaisRetorno ===")
    
    base_urls = [
        'https://maisretorno.com',
        'https://api.maisretorno.com',
        'https://www.maisretorno.com/api'
    ]
    
    results = {}
    
    for url in base_urls:
        print(f"\nTestando: {url}")
        
        try:
            response = requests.get(url, timeout=10, allow_redirects=True)
            print(f"Status: {response.status_code}")
            print(f"URL final: {response.url}")
            
            # Verificar se há menções de API no content-type ou headers
            content_type = response.headers.get('content-type', '')
            print(f"Content-Type: {content_type}")
            
            # Verificar se responde JSON
            if 'application/json' in content_type:
                try:
                    json_data = response.json()
                    print(f"Resposta JSON: {json_data}")
                except:
                    print("Não foi possível parsear JSON")
            
            results[url] = {
                'status': response.status_code,
                'final_url': response.url,
                'content_type': content_type
            }
            
        except Exception as e:
            print(f"Erro: {str(e)}")
            results[url] = {'error': str(e)}
    
    return results

In [ ]:
maisretorno_results = investigate_maisretorno()

## 7. Consolidação dos Resultados

In [ ]:
# Consolidar todos os resultados
investigation_summary = {
    'bcb_api': bcb_results,
    'tesouro_api': tesouro_datasets,
    'cvm_api': cvm_results,
    'news_sources': news_results,
    'maisretorno_investigation': maisretorno_results
}

print("\n" + "="*60)
print("RESUMO EXECUTIVO DA INVESTIGAÇÃO")
print("="*60)

# Análise BCB
print("\n1. API BCB (Banco Central):")
bcb_status = "✅ DISPONÍVEL" if any(bcb_results.values()) else "❌ INDISPONÍVEL"
print(f"   Status: {bcb_status}")
if bcb_results:
    available_series = [k for k, v in bcb_results.items() if v is not None]
    print(f"   Séries disponíveis: {', '.join(available_series)}")

# Análise CVM
print("\n2. API CVM (Dados de Fundos):")
cvm_available = any(r.get('status') == 200 for r in cvm_results.values() if isinstance(r, dict))
cvm_status = "✅ DISPONÍVEL" if cvm_available else "⚠️  VERIFICAÇÃO NECESSÁRIA"
print(f"   Status: {cvm_status}")

# Análise Notícias
print("\n3. Fontes de Notícias:")
news_available = any('error' not in r for r in news_results.values())
news_status = "✅ DISPONÍVEL" if news_available else "❌ INDISPONÍVEL"
print(f"   Status: {news_status}")

# Análise MaisRetorno
print("\n4. MaisRetorno API:")
maisretorno_api = any('application/json' in r.get('content_type', '') for r in maisretorno_results.values() if isinstance(r, dict))
maisretorno_status = "✅ API ENCONTRADA" if maisretorno_api else "❌ API NÃO ENCONTRADA - SCRAPING NECESSÁRIO"
print(f"   Status: {maisretorno_status}")

print("\n" + "="*60)

## 8. Conclusões e Recomendações Técnicas

### Arquitetura Recomendada

**Stack Validada:**
- ✅ Python + pandas para análise
- ✅ requests para APIs oficiais
- ✅ SQLite para persistência
- ✅ ReportLab para PDF
- ⚠️ selenium para scraping (fallback)

**Pipeline de Dados:**
1. **APIs Oficiais (Prioridade):** BCB para benchmarks, CVM para fundos
2. **RSS Feeds:** Google News para notícias Nubank
3. **Scraping (Fallback):** MaisRetorno se APIs não cobrirem dados necessários

**Próximos Passos:**
1. Implementar cliente BCB para SELIC/CDI
2. Testar download de dados CVM
3. Implementar parser RSS para notícias
4. Definir schema SQLite
5. Prototipar scraping MaisRetorno como fallback

### Riscos e Mitigações

- **Risco:** APIs podem ter rate limits
- **Mitigação:** Implementar cache local e respeitar limites

- **Risco:** Dados CVM podem ser grandes
- **Mitigação:** Filtrar apenas fundos de interesse

- **Risco:** Scraping pode quebrar
- **Mitigação:** APIs oficiais como fonte primária